In [ ]:
# 1. IMPORT CÁC THƯ VIỆN CẦN THIẾT
import cv2
import mediapipe as mp
import numpy as np

print("Đã import thư viện thành công!")

# 2. KHỞI TẠO MEDIAPIPE
mp_holistic = mp.solutions.holistic # "Giải pháp" Holistic
mp_drawing = mp.solutions.drawing_utils # Tiện ích để vẽ

# 3. KHỞI TẠO ĐỐI TƯỢNG HOLISTIC
# min_detection_confidence: Ngưỡng tin cậy để phát hiện (0.5 = 50%)
# min_tracking_confidence: Ngưỡng tin cậy để theo dõi (0.5 = 50%)
holistic = mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# 4. KHỞI TẠO WEBCAM (dùng OpenCV)
cap = cv2.VideoCapture(0) # Số 0 là webcam mặc định

# 5. VÒNG LẶP XỬ LÝ
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Bỏ qua frame camera...")
        continue

    # Lật ảnh cho giống chế độ selfie và TỐI ƯU HÓA
    # 1. Lật ảnh (flip)
    image = cv2.flip(frame, 1)
    
    # 2. Chuyển đổi màu BGR (OpenCV) sang RGB (MediaPipe)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 3.Đánh dấu ảnh là "không thể ghi" để tăng tốc độ
    image_rgb.flags.writeable = False
    
    # 4. CHẠY MEDIAPIPE
    results = holistic.process(image_rgb)
    
    # 5.Cho phép ghi lại ảnh để vẽ
    image_rgb.flags.writeable = True
    
    # 6. Chuyển lại sang BGR để OpenCV hiển thị
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    # 7. VẼ CÁC ĐIỂM MỐC (LANDMARKS) LÊN ẢNH
    
    # Vẽ mặt (với các đường nối)
    mp_drawing.draw_landmarks(
        image_bgr, 
        results.face_landmarks, 
        mp_holistic.FACEMESH_CONTOURS,
        mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), # Chấm
        mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1) # Nối
    )
    
    # Vẽ Dáng (Pose)
    mp_drawing.draw_landmarks(
        image_bgr, 
        results.pose_landmarks, 
        mp_holistic.POSE_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
    )
    
    # Vẽ Bàn tay (Trái và Phải)
    mp_drawing.draw_landmarks(
        image_bgr, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
    mp_drawing.draw_landmarks(
        image_bgr, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

    # 8. HIỂN THỊ HÌNH ẢNH
    # Một cửa sổ mới sẽ bật lên (pop-up)
    cv2.imshow('MediaPipe Holistic - Nhan phim Q de thoat', image_bgr)

    # 9. THOÁT (Nhấn phím 'q')
    # cv2.waitKey(5) nghĩa là chờ 5ms, & 0xFF là mặt nạ
    if cv2.waitKey(5) & 0xFF == ord('q'):
        break

# 10. DỌN DẸP
holistic.close()
cap.release()
cv2.destroyAllWindows()

# In ra để báo đã xong (nếu thoát vòng lặp)
print("Đã đóng camera và giải phóng tài nguyên.")

Đã import thư viện thành công!
Đã đóng camera và giải phóng tài nguyên.


In [13]:
import json
from collections import Counter
import os

# --- 1. CÀI ĐẶT THÔNG SỐ ---
JSON_FILE_PATH = 'MS-ASL/MSASL_train.json' # Đảm bảo file này ở cùng thư mục
TOP_K = 100 # Chúng ta muốn lấy 100 từ

# --- 2. ĐỌC VÀ ĐẾM TỪ (LABEL) ---
print(f"Đang đọc file {JSON_FILE_PATH}...")
word_counts = Counter()
total_entries = 0

# Mở file JSON
with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)
    total_entries = len(data)
    
    # Lặp qua tất cả 25.000+ mục và đếm nhãn (label)
    for entry in data:
        label = entry['label']
        word_counts[label] += 1

print(f"Đã phân tích xong {total_entries} mục.")
print(f"Tìm thấy tổng cộng {len(word_counts)} từ vựng (label) duy nhất.")

# --- 3. LẤY 100 TỪ PHỔ BIẾN NHẤT ---

# Lấy 100 từ (label) và số lần xuất hiện của chúng
top_k_words = word_counts.most_common(TOP_K)

# In ra 10 từ đầu tiên cho bạn xem
print("\n--- 10 TỪ PHỔ BIẾN NHẤT ---")
for word, count in top_k_words[:10]:
    print(f"{word}: {count} lần")

# Tạo một "Set" (tập hợp) chứa 100 từ này để tra cứu nhanh
# Đây là "Từ điển MS-ASL 100" của bạn
top_k_word_set = {word for word, count in top_k_words}
print(f"\nĐã tạo bộ MS-ASL 100 (với {len(top_k_word_set)} từ).")


# --- 4. TẠO FILE JSON MỚI (Lọc lại) ---
# Đây là bước quan trọng: Tạo 1 file JSON mới CHỈ chứa 
# các video thuộc 100 từ phổ biến nhất

ms_asl_100_data = []
for entry in data:
    if entry['label'] in top_k_word_set:
        ms_asl_100_data.append(entry)

# Lưu file JSON đã lọc
output_filename = 'MS-ASL-100.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(ms_asl_100_data, f, indent=4)

print(f"\nĐã lưu thành công bộ dữ liệu MS-ASL 100 vào file: {output_filename}")
print(f"Số lượng video trong bộ MS-ASL 100: {len(ms_asl_100_data)}")

Đang đọc file MS-ASL/MSASL_train.json...
Đã phân tích xong 16054 mục.
Tìm thấy tổng cộng 1000 từ vựng (label) duy nhất.

--- 10 TỪ PHỔ BIẾN NHẤT ---
3: 57 lần
1: 54 lần
8: 53 lần
2: 50 lần
7: 50 lần
14: 50 lần
6: 48 lần
13: 48 lần
15: 48 lần
17: 48 lần

Đã tạo bộ MS-ASL 100 (với 100 từ).

Đã lưu thành công bộ dữ liệu MS-ASL 100 vào file: MS-ASL-100.json
Số lượng video trong bộ MS-ASL 100: 3865
